# Polyphase Interpolation**Source**: `grdl/example/interpolation/polyphaseinterpolation.py`## Learning Objectives- Understand polyphase filter bank structure- Resample signals at arbitrary fractional rates- Compare Kaiser vs Remez prototype filter designs- Preserve signal bandwidth during resampling## Theory: Polyphase Interpolation**Problem**: Resample a signal from sample rate $f_s^{\text{in}}$ to $f_s^{\text{out}}$ while preserving bandwidth.**Naive approach** (bad): Upsample → lowpass filter → downsample  **Polyphase approach** (good): Fractional delay filter bank with efficient indexing**How it works**:1. Design a prototype lowpass filter (Kaiser or Remez)2. Decompose into $P$ polyphase branches (phases)3. For each output sample, select the phase closest to the fractional delay4. Apply that phase's filter taps**Key parameters**:- **kernel_length**: Number of taps per phase (16 = good balance)- **num_phases**: Polyphase bank size (2048 = high precision)- **prototype**: Filter design method ('kaiser' or 'remez')**Kaiser vs Remez**:- **Kaiser**: Parameterized window, fast design, moderate stopband- **Remez**: Equiripple optimal, slower design, better stopband rejection**Use cases**:- SAR image formation (range/azimuth resampling)- Doppler compensation- Arbitrary sample rate conversion## Data SetupThis example uses a **synthetic LFM chirp** — no external data required.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom grdl.interpolation import PolyphaseInterpolator

In [ ]:
# Generate a complex LFM (Linear Frequency Modulated) chirpchirp_rate = 250.0  # Hz/s (frequency sweeps linearly over time)fs_in = 5e5  # Input sample rate: 500 kHzn_in = int(fs_in)x_in = np.arange(n_in) / fs_insignal_in = np.exp(1j * 2 * np.pi * chirp_rate * x_in ** 2).astype(np.complex64)print(f"Input signal:")print(f"  Sample rate: {fs_in / 1e3:.1f} kHz")print(f"  Samples: {n_in:,}")print(f"  Duration: {n_in / fs_in:.3f} s")print(f"  Chirp rate: {chirp_rate} Hz/s")print(f"  Frequency range: [0, {chirp_rate * (n_in / fs_in):.1f}] Hz")

In [ ]:
# Define output sample grid (different rate)fs_out = 2.123e5  # Output sample rate: 212.3 kHz (arbitrary fractional rate)n_out = int(fs_out)x_out = np.arange(n_out) / fs_outprint(f"\nOutput sample grid:")print(f"  Sample rate: {fs_out / 1e3:.1f} kHz")print(f"  Samples: {n_out:,}")print(f"  Duration: {n_out / fs_out:.3f} s")print(f"  Rate ratio: {fs_out / fs_in:.4f} (resampling factor)")

In [ ]:
# Resample using Kaiser prototype (default)interp_kaiser = PolyphaseInterpolator(    kernel_length=16,  # 16 taps per phase (moderate quality)    num_phases=2048,   # 2048 phases (high fractional delay precision)    prototype='kaiser',)y_kaiser = interp_kaiser(x_in, signal_in, x_out)print(f"\nKaiser interpolation:")print(f"  Output samples: {len(y_kaiser):,}")print(f"  Output shape: {y_kaiser.shape}")print(f"  Output dtype: {y_kaiser.dtype}")

In [ ]:
# Resample using Remez prototype (better stopband rejection)interp_remez = PolyphaseInterpolator(    kernel_length=16,    num_phases=2048,    prototype='remez',)y_remez = interp_remez(x_in, signal_in, x_out)print(f"\nRemez interpolation:")print(f"  Output samples: {len(y_remez):,}")print(f"  Kernel length: {interp_remez.kernel_length}")print(f"  Num phases: {interp_remez.num_phases}")

In [ ]:
# Visualize time-domain comparisonfig, axes = plt.subplots(2, 1, figsize=(12, 8))# Time domain (first 500 input samples, 200 output samples)axes[0].set_title('Time Domain: Input vs Resampled (Kaiser vs Remez)', fontsize=14)axes[0].plot(x_in[:500], np.real(signal_in[:500]), label=f'Input ({fs_in/1e3:.0f} kHz)', alpha=0.7, linewidth=1)axes[0].plot(x_out[:200], np.real(y_kaiser[:200]), '--', label=f'Kaiser ({fs_out/1e3:.1f} kHz)', linewidth=1.5)axes[0].plot(x_out[:200], np.real(y_remez[:200]), ':', label=f'Remez ({fs_out/1e3:.1f} kHz)', linewidth=1.5)axes[0].legend(loc='upper right')axes[0].set_xlabel('Time (s)')axes[0].set_ylabel('Real Part')axes[0].grid(True, alpha=0.3)# Spectrum comparisonin_freq = (np.arange(n_in) / n_in - 0.5) * fs_inin_spec = np.fft.fftshift(np.fft.fft(signal_in))out_freq_k = (np.arange(len(y_kaiser)) / len(y_kaiser) - 0.5) * fs_outout_spec_k = np.fft.fftshift(np.fft.fft(y_kaiser))out_freq_r = (np.arange(len(y_remez)) / len(y_remez) - 0.5) * fs_outout_spec_r = np.fft.fftshift(np.fft.fft(y_remez))axes[1].set_title('Spectrum: Bandwidth Preservation', fontsize=14)axes[1].plot(in_freq / 1e3, np.abs(in_spec), label='Input', alpha=0.5, linewidth=1)axes[1].plot(out_freq_k / 1e3, np.abs(out_spec_k), '--', label='Kaiser', linewidth=1.5)axes[1].plot(out_freq_r / 1e3, np.abs(out_spec_r), ':', label='Remez', linewidth=1.5)axes[1].legend(loc='upper right')axes[1].set_xlabel('Frequency (kHz)')axes[1].set_ylabel('Magnitude')axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Physical Interpretation**Time domain**: Both Kaiser and Remez interpolators produce smooth resampled signals that closely follow the input chirp. The polyphase approach avoids phase discontinuities that plague naive resampling.**Frequency domain**: The spectrum is preserved — no aliasing, no excessive high-frequency attenuation. The signal bandwidth is maintained after resampling.**Kaiser vs Remez**: - Kaiser is faster to design and adequate for most applications- Remez provides better stopband rejection (sharper cutoff) at the cost of design time- For this chirp example, both perform nearly identically**Computational efficiency**: Polyphase interpolation is $O(N \cdot L)$ where $N$ = output samples, $L$ = kernel length. Much faster than upsample + filter + downsample.**Fractional delay precision**: With 2048 phases, the fractional delay quantization error is $< 1/(2 \times 2048) = 0.024\%$ of a sample — effectively continuous.## When to Use Polyphase Interpolation**Ideal for**:- SAR range/azimuth resampling (change pixel spacing)- Doppler compensation (time-varying sample rate)- Multi-rate signal processing (decimation, interpolation, fractional resampling)**Not ideal for**:- Upsampling by integer factors (use efficient FFT-based upsampling)- Very short signals (overhead of polyphase bank design)## Summary**GRDL patterns demonstrated**:- ✅ `PolyphaseInterpolator` — bandwidth-preserving arbitrary rate conversion- ✅ Kaiser vs Remez prototype comparison- ✅ Fractional delay filter bank**Key insight**: Polyphase interpolation is the gold standard for SAR resampling — it preserves signal bandwidth while allowing arbitrary fractional rate changes.